# NB14 — Third architecture + better signals: testing the unifying claim's generality

The EffNet result (NB13) showed the single-factor structure is PARTIAL: calibration + explanation
faithfulness (intrinsic signals) share the competence factor on both Xception and EffNet (PC1=87%
on EffNet for those two), but the reference-divergence MONITORING signals (KS/Wasserstein) decouple
from competence on EffNet. This notebook tests two things:

1. **Third architecture** (CLIP ViT foundation model, fallback F3Net): does the intrinsic-signal
   single-factor hold on a THIRD, architecturally-different backbone? CNN+CNN+ViT = strong generality.
2. **Reference-free monitoring signals**: the KS/Wasserstein signals depend on a fixed in-domain
   reference (the architecture-fragile part). We add reference-free uncertainty signals (predictive
   entropy, score dispersion, mean-max-confidence, fraction-near-0.5) that measure the detector's OWN
   uncertainty. Do THESE track competence on all architectures?

Outputs: unified_trust_signals_<arch>.csv with intrinsic + reference-free signals, plus a
three-architecture comparison of the intrinsic-signal single-factor.
Per-generator checkpointed.


## Cell 1 — Setup + load third architecture (CLIP, fallback F3Net)

In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, sys, glob, subprocess, importlib, importlib.util
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
DFB  = f"{REPO}/external/DeepfakeBench"
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"/content/drive/MyDrive/CDTS_Research/{f}"): subprocess.run(f'cp "/content/drive/MyDrive/CDTS_Research/{f}" /root/{f}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)
subprocess.run("pip -q install efficientnet_pytorch timm einops kornia simplejson", shell=True)

def load_arch(backbone_name, ckpt_name):
    for k in list(sys.modules.keys()):
        if k.startswith("detectors") or k.startswith("networks") or k=="metrics" or k.startswith("metrics.") or k=="inference":
            del sys.modules[k]
    for p in (f"{DFB}/training", DFB, f"{REPO}/src"):
        if p in sys.path: sys.path.remove(p)
    sys.path.insert(0, DFB); sys.path.insert(0, f"{DFB}/training"); sys.path.append(f"{REPO}/src")
    spec=importlib.util.spec_from_file_location("inference",f"{REPO}/src/inference.py")
    inf=importlib.util.module_from_spec(spec); sys.modules["inference"]=inf; spec.loader.exec_module(inf)
    ckpt=f"{REPO}/weights/train_on_fs/{ckpt_name}"
    m,dev,info=inf.load_detector(dfb_root=DFB, backbone_name=backbone_name, ckpt_path=ckpt)
    m.eval(); return m,dev,info,inf

# Try CLIP first (ViT foundation model - most different from Xception/EffNet)
ARCH=None
for bn, ck, tag in [("clip","clip.pth","CLIP-ViT"), ("f3net","f3net.pth","F3Net")]:
    try:
        model,device,info,inference = load_arch(bn, ck)
        ARCH=tag; ARCH_KEY=bn
        print(f"LOADED {tag}: {info}")
        break
    except Exception as e:
        print(f"{tag} failed: {str(e)[:120]}")
assert ARCH, "no third architecture could be loaded"
# CLIP ViT expects 224x224; Xception/EffNet/F3Net use 256x256
ARCH_RES = 224 if ARCH_KEY=="clip" else 256
print(f"\n>>> Using {ARCH} as the third architecture (input resolution {ARCH_RES}x{ARCH_RES})")
# inspect feature/output structure for Grad-CAM planning
import torch
x=torch.randn(1,3,ARCH_RES,ARCH_RES).to(device)
with torch.no_grad(): out=model({'image':x},inference=True)
print("output keys:", list(out.keys()))
if 'feat' in out: print("feat shape:", out['feat'].shape)
convs=[n for n,m in model.backbone.named_modules() if 'Conv2d' in type(m).__name__]
print(f"conv layers: {len(convs)} (last 3: {convs[-3:] if convs else 'NONE - likely ViT'})")

Mounted at /content/drive


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LOADED CLIP-ViT: {'missing': 0, 'unexpected': 0}

>>> Using CLIP-ViT as the third architecture (input resolution 224x224)
output keys: ['cls', 'prob', 'feat']
feat shape: torch.Size([1, 768])
conv layers: 1 (last 3: ['embeddings.patch_embedding'])


In [6]:
# ============================================================================
# NB14 Cell 1b (VERIFY) — run after Cell 1, before the long scoring cell.
# Confirms CLIP scores correctly at 224x224 on a handful of real frames,
# and checks whether Grad-CAM/faithfulness is available (ViT likely NaN).
# ============================================================================
import torch, numpy as np, glob, os, zipfile, shutil, cv2

# 1) confirm the test forward pass works at ARCH_RES (already ran in Cell 1, but re-confirm)
x=torch.randn(1,3,ARCH_RES,ARCH_RES).to(device)
with torch.no_grad(): out=model({'image':x},inference=True)
print(f"forward pass at {ARCH_RES}x{ARCH_RES}: OK")
print(f"  output keys: {list(out.keys())}")
print(f"  prob shape: {out['prob'].shape if hasattr(out['prob'],'shape') else 'scalar'}")
print(f"  cls/logits shape: {out['cls'].shape}")

# 2) score a few REAL frames + a few FAKE frames from one generator to sanity-check scoring
REPO="/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
SHORTCUT="/content/drive/MyDrive/CDTS_Research/DF40"
# grab simswap (high-competence on Xcep) as a quick test
testzip=None
for c in [f"{REPO}/data/df40_core/test/simswap.zip", f"{SHORTCUT}/simswap.zip"]:
    if os.path.exists(c) and os.path.getsize(c)>1e6: testzip=c; break

MEAN=torch.tensor([0.5,0.5,0.5]).view(1,3,1,1).to(device)
STD=torch.tensor([0.5,0.5,0.5]).view(1,3,1,1).to(device)
def _load(p,res):
    im=cv2.imread(p)[:,:,::-1]; im=cv2.resize(im,(res,res)); return im.astype(np.float32)/255.0
def _tt(im): return (torch.from_numpy(im).permute(2,0,1).unsqueeze(0).to(device)-MEAN)/STD

if testzip:
    td="/content/clip_test"; os.makedirs(td,exist_ok=True)
    with zipfile.ZipFile(testzip) as z: z.extractall(td)
    pngs=sorted(glob.glob(f"{td}/**/*.png",recursive=True))[:8]
    print(f"\nScoring {len(pngs)} fake frames from simswap at {ARCH_RES}x{ARCH_RES}:")
    with torch.no_grad():
        for p in pngs[:8]:
            o=model({'image':_tt(_load(p,ARCH_RES))},inference=True)
            pf=float(o['prob']) if o['prob'].dim()==0 else float(o['prob'][0])
            print(f"  prob_fake={pf:.3f}")
    shutil.rmtree(td,ignore_errors=True)
    print("  (these are FAKE frames; a competent detector should give HIGH prob_fake)")
else:
    print("\n(simswap zip not found for quick test; will score in main cell)")

# 3) Grad-CAM availability check (the FAKE_IDX probe + spatial conv)
print(f"\nGrad-CAM / faithfulness availability:")
print(f"  CAM_OK = {CAM_OK}")
if CAM_OK:
    print(f"  target layer: {CAM_NAME}")
    print(f"  -> faithfulness WILL be computed")
else:
    print(f"  -> CLIP is a ViT with no usable spatial conv map")
    print(f"  -> faithfulness will be NaN for CLIP (expected); we rely on calibration +")
    print(f"     reference-free signals (entropy, dispersion) for CLIP's trust-signal vector.")
    print(f"  -> NOTE: this means CLIP can't contribute to the intrinsic-signal single-factor")
    print(f"     test that uses faithfulness. CLIP still tests whether CALIBRATION and the")
    print(f"     reference-free monitoring signals track competence on a transformer.")
print("\n>>> If prob_fake values look reasonable (high for these fake frames), proceed to scoring.")

forward pass at 224x224: OK
  output keys: ['cls', 'prob', 'feat']
  prob shape: torch.Size([1])
  cls/logits shape: torch.Size([1, 2])

Scoring 8 fake frames from simswap at 224x224:
  prob_fake=1.000
  prob_fake=1.000
  prob_fake=1.000
  prob_fake=1.000
  prob_fake=1.000
  prob_fake=1.000
  prob_fake=1.000
  prob_fake=1.000
  (these are FAKE frames; a competent detector should give HIGH prob_fake)

Grad-CAM / faithfulness availability:


NameError: name 'CAM_OK' is not defined

## Cell 2 — Signal machinery: intrinsic (faithfulness) + reference-free uncertainty

Reference-free signals (no in-domain reference needed, so architecture-portable):
- **predictive_entropy**: mean binary entropy of scores (classic uncertainty/OOD proxy)
- **score_dispersion**: std of scores
- **mean_max_conf**: mean of max(p, 1-p) (confidence; Hendrycks-Gimpel style)
- **frac_uncertain**: fraction of scores in [0.4, 0.6]

Faithfulness: same saturation-free per-region rank metric (if backbone has a conv map; for ViT we
attempt the last conv/projection, else mark faithfulness NaN and rely on intrinsic calibration +
reference-free signals).


In [7]:
import torch, numpy as np, cv2
import torch.nn.functional as F
from scipy.stats import spearmanr
MEAN=torch.tensor([0.5,0.5,0.5]).view(1,3,1,1).to(device)
STD=torch.tensor([0.5,0.5,0.5]).view(1,3,1,1).to(device)
def load_img(p):
    im=cv2.imread(p)[:,:,::-1];im=cv2.resize(im,(ARCH_RES,ARCH_RES));return im.astype(np.float32)/255.0
def to_tensor(im):return (torch.from_numpy(im).permute(2,0,1).unsqueeze(0).to(device)-MEAN)/STD

# ---- determine the FAKE class index (don't assume index 1) ----
# Probe with a known-fake-ish input is unreliable; instead check head output dim + use the
# convention that 'prob' (returned by inference=True) is P(fake). We pick the logit index whose
# softmax matches 'prob' on a sample.
@torch.no_grad()
def _fake_index_probe(im):
    x=to_tensor(im)
    out=model({'image':x},inference=True)
    prob_fake=float(out['prob']) if out['prob'].dim()==0 else float(out['prob'][0])
    logits=out['cls'][0]
    sm=torch.softmax(logits,dim=0).cpu().numpy()
    # the class whose softmax prob is closest to prob_fake is the fake class
    idx=int(np.argmin(np.abs(sm-prob_fake)))
    return idx, prob_fake, sm
# (probe deferred until we have a real frame in Cell on scoring; default to 1, re-checked below)
FAKE_IDX = 1  # will be re-probed on first real frame

# ---- pick a Grad-CAM target conv that is actually SPATIAL (H>1,W>1) ----
def _last_spatial_conv():
    if not hasattr(model,'backbone'): return None,None
    cand=[(n,m) for n,m in model.backbone.named_modules() if isinstance(m,torch.nn.Conv2d)]
    if not cand: return None,None
    # forward once, hook each candidate, keep those whose output is spatial
    acts={}
    handles=[]
    for n,m in cand:
        handles.append(m.register_forward_hook(lambda mod,i,o,nm=n: acts.__setitem__(nm,o.shape)))
    with torch.no_grad():
        model({'image':torch.randn(1,3,ARCH_RES,ARCH_RES).to(device)},inference=True)
    for h in handles: h.remove()
    spatial=[(n,m) for n,m in cand if n in acts and len(acts[n])==4 and acts[n][2]>1 and acts[n][3]>1]
    if not spatial: return None,None
    name,mod=spatial[-1]  # last spatial conv
    return name,mod

CAM_NAME, TARGET = _last_spatial_conv()
CAM_OK = TARGET is not None
print(f"Grad-CAM: {'enabled, target='+CAM_NAME if CAM_OK else 'DISABLED (no spatial conv map - ViT/transformer). faithfulness will be NaN'}")

def grad_cam(im):
    if not CAM_OK: return None
    acts={};grads={}
    h1=TARGET.register_forward_hook(lambda m,i,o:acts.__setitem__('a',o))
    h2=TARGET.register_full_backward_hook(lambda m,gi,go:grads.__setitem__('g',go[0]))
    x=to_tensor(im).requires_grad_(True);out=model({'image':x},inference=False)
    model.zero_grad();out['cls'][0,FAKE_IDX].backward()
    A=acts['a'][0];G=grads['g'][0]
    if A.dim()!=3 or A.shape[1]<2 or A.shape[2]<2:
        h1.remove();h2.remove();return None
    w=G.mean(dim=(1,2))
    cam=F.relu((w.view(-1,1,1)*A).sum(0));cam=cam-cam.min();cam=cam/(cam.max()+1e-8)
    Hc,Wc=cam.shape
    cam=F.interpolate(cam.view(1,1,Hc,Wc),size=(ARCH_RES,ARCH_RES),mode='bilinear',align_corners=False)[0,0]
    h1.remove();h2.remove();return cam.detach().cpu().numpy()
@torch.no_grad()
def prob_fake_of(im):
    out=model({'image':to_tensor(im)},inference=True)
    return float(out['prob']) if out['prob'].dim()==0 else float(out['prob'][0])
def faithfulness_rankcorr(im, grid=8):
    cam=grad_cam(im)
    if cam is None: return np.nan
    Hc,Wc=cam.shape
    ph,pw=max(Hc//grid,1),max(Wc//grid,1)   # guard against 0 (small feature maps)
    blurred=cv2.GaussianBlur(im,(0,0),sigmaX=11);base=prob_fake_of(im)
    imp=[];eff=[]
    gy_range=range(0,Hc,ph); gx_range=range(0,Wc,pw)
    for y0 in gy_range:
        for x0 in gx_range:
            y1=min(y0+ph,Hc); x1=min(x0+pw,Wc)
            imp.append(float(cam[y0:y1,x0:x1].mean()))
            tmp=im.copy()
            # map feature-grid cell to image coords (cam is already 256x256 here)
            tmp[y0:y1,x0:x1,:]=blurred[y0:y1,x0:x1,:]
            eff.append(base-prob_fake_of(tmp))
    imp=np.array(imp);eff=np.array(eff)
    if imp.std()<1e-6 or eff.std()<1e-6 or len(imp)<4: return np.nan
    return float(spearmanr(imp,eff).statistic)

def reference_free_signals(p):
    p=np.clip(p,1e-7,1-1e-7)
    ent=float(np.mean(-p*np.log2(p)-(1-p)*np.log2(1-p)))
    disp=float(np.std(p))
    maxconf=float(np.mean(np.maximum(p,1-p)))
    frac_unc=float(np.mean((p>=0.4)&(p<=0.6)))
    return {'entropy':ent,'score_dispersion':disp,'mean_max_conf':maxconf,'frac_uncertain':frac_unc}
print("machinery ready (FAKE_IDX will be re-probed on first real frame in scoring cell)")

Grad-CAM: enabled, target=embeddings.patch_embedding
machinery ready (FAKE_IDX will be re-probed on first real frame in scoring cell)


In [8]:
# ============================================================================
# NB14 — CLIP Grad-CAM override. RUN THIS after Cell 2, before scoring (Cell 3).
# Cell 2 wrongly picked 'embeddings.patch_embedding' (CLIP's INPUT patch-embed
# conv) as the Grad-CAM target. That layer is pre-attention and gives meaningless
# faithfulness. For a ViT, honest Grad-CAM faithfulness is not available this way,
# so we disable it and rely on calibration + reference-free signals.
# ============================================================================
if ARCH_KEY == "clip":
    CAM_OK = False
    CAM_NAME = None
    print("CLIP: Grad-CAM DISABLED (patch-embedding is pre-attention; not a valid")
    print("      saliency target for a ViT). faithfulness_rankcorr will be NaN for CLIP.")
    print("      CLIP's trust-signal vector = calibration (ECE) + reference-free signals")
    print("      (entropy, dispersion, mean_max_conf, frac_uncertain).")
    print()
    print("      This is the honest choice: meaningful Grad-CAM faithfulness on a")
    print("      transformer needs attention-rollout or attention-flow methods, which")
    print("      are a different instrument than the conv Grad-CAM used for CNNs;")
    print("      mixing them would not be a like-for-like faithfulness comparison.")
else:
    print(f"{ARCH_KEY}: keeping Grad-CAM as detected (CAM_OK={CAM_OK}, target={CAM_NAME})")

# sanity: show what CLIP's feat looks like (for the record / future attention-based work)
import torch
with torch.no_grad():
    o = model({'image': torch.randn(1,3,ARCH_RES,ARCH_RES).to(device)}, inference=True)
if 'feat' in o:
    print(f"\n(for reference) CLIP 'feat' shape: {tuple(o['feat'].shape)}")
    print("  if this were spatial (B,C,H,W with H,W>1), attention-free CAM might be possible;")
    print("  CLIP feat is typically a pooled (B, D) embedding -> confirms no spatial saliency.")

CLIP: Grad-CAM DISABLED (patch-embedding is pre-attention; not a valid
      saliency target for a ViT). faithfulness_rankcorr will be NaN for CLIP.
      CLIP's trust-signal vector = calibration (ECE) + reference-free signals
      (entropy, dispersion, mean_max_conf, frac_uncertain).

      This is the honest choice: meaningful Grad-CAM faithfulness on a
      transformer needs attention-rollout or attention-flow methods, which
      are a different instrument than the conv Grad-CAM used for CNNs;
      mixing them would not be a like-for-like faithfulness comparison.

(for reference) CLIP 'feat' shape: (1, 768)
  if this were spatial (B,C,H,W with H,W>1), attention-free CAM might be possible;
  CLIP feat is typically a pooled (B, D) embedding -> confirms no spatial saliency.


## Cell 3 — Score generators + compute all signals (checkpointed)

In [9]:
import os, glob, zipfile, shutil
import pandas as pd, numpy as np
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
SHORTCUT="/content/drive/MyDrive/CDTS_Research/DF40";LOCALCORE=f"{REPO}/data/df40_core/test"
OUT=f"{REPO}/reports/calibration/unified_trust_signals_{ARCH_KEY}.csv"
CDF_REAL=f"{REPO}/data/frames/Celeb-DF-v2"

def use_inference_env():
    for k in list(sys.modules.keys()):
        if k in ("metrics","calibration","data_prep") or k.startswith("metrics."): del sys.modules[k]
    for p in (f"{DFB}/training", DFB, f"{REPO}/src"):
        if p in sys.path: sys.path.remove(p)
    sys.path.insert(0, DFB); sys.path.insert(0, f"{DFB}/training"); sys.path.append(f"{REPO}/src")
def use_calibration_env():
    for k in list(sys.modules.keys()):
        if k in ("metrics","calibration","data_prep") or k.startswith("metrics."): del sys.modules[k]
    for p in (f"{DFB}/training", DFB, f"{REPO}/src"):
        if p in sys.path: sys.path.remove(p)
    sys.path.insert(0, f"{REPO}/src")

NEED = pd.read_csv(f"{REPO}/reports/calibration/unified_trust_signals.csv")['method'].tolist()
done=set(pd.read_csv(OUT)['method']) if os.path.exists(OUT) else set()
real_index={"/".join(fp.split("/")[-2:]):fp for fp in glob.glob(f"{CDF_REAL}/**/frames/**/*.png",recursive=True)}
def get_zip(m):
    for c in [f"{LOCALCORE}/{m}.zip",f"{SHORTCUT}/{m}.zip",f"{SHORTCUT}/{m.upper()}.zip"]:
        if os.path.exists(c) and os.path.getsize(c)>1e6: return c
    return None

for method in NEED:
    if method in done: continue
    print(f"=== {method} ===")
    sp=f"{REPO}/reports/scores/{ARCH_KEY}_df40_{method}.parquet"
    jpath=f"{REPO}/data/df40/dataset_json/{method}_cdf.json"
    fdir=None; faiths=[]
    if not os.path.exists(sp):
        zp=get_zip(method)
        if not zp or not os.path.exists(jpath): print(f"  cannot score, skip\n"); continue
        fdir=f"/content/a3_{method}";os.makedirs(fdir,exist_ok=True)
        try:
            with zipfile.ZipFile(zp) as z: z.extractall(fdir)
        except Exception as e: print(f"  unzip fail: {str(e)[:50]}\n");shutil.rmtree(fdir,ignore_errors=True);continue
        use_inference_env(); import data_prep as dp
        fake_index={"/".join(fp.split("/")[-2:]):fp for fp in glob.glob(f"{fdir}/**/*.png",recursive=True)}
        df=dp.build_manifest_from_json(f"{method}_cdf",jpath,frames_root=None)
        df['frame_path']=df.apply(lambda r: fake_index.get("/".join(str(r['frame_path']).split("/")[-2:])) if r['label']==1 else real_index.get("/".join(str(r['frame_path']).split("/")[-2:])),axis=1)
        df=df[df['frame_path'].notna()].reset_index(drop=True)
        if df['label'].nunique()<2: print("  one label, skip\n");shutil.rmtree(fdir,ignore_errors=True);continue
        sc=inference.score_manifest(model,device,df,batch_size=64,res=ARCH_RES,verbose=False); sc.to_parquet(sp,index=False)
        print(f"  scored {len(sc)}")
    else:
        sc=pd.read_parquet(sp); print(f"  loaded {len(sc)}")
    # faithfulness (if conv available)
    if CAM_OK:
        if fdir is None:
            zp=get_zip(method)
            if zp:
                fdir=f"/content/a3_{method}";os.makedirs(fdir,exist_ok=True)
                try:
                    with zipfile.ZipFile(zp) as z: z.extractall(fdir)
                except Exception: fdir=None
        if fdir:
            use_inference_env()
            fake_index={"/".join(fp.split("/")[-2:]):fp for fp in glob.glob(f"{fdir}/**/*.png",recursive=True)}
            global FAKE_IDX
            _probed=False
            _fail=0
            for _,r in sc[sc.label==1].sample(min((sc.label==1).sum(),90),random_state=42).iterrows():
                rp=fake_index.get("/".join(str(r['frame_path']).split("/")[-2:]))
                if not rp and os.path.exists(str(r['frame_path'])): rp=r['frame_path']
                if not rp or not os.path.exists(rp): continue
                # re-probe the fake class index once, on the first valid real frame
                if not _probed:
                    try:
                        _idx,_pf,_sm=_fake_index_probe(load_img(rp))
                        if _idx!=FAKE_IDX:
                            print(f"  [probe] FAKE_IDX {FAKE_IDX} -> {_idx} (softmax {np.round(_sm,3)}, prob_fake {_pf:.3f})")
                            FAKE_IDX=_idx
                        _probed=True
                    except Exception as e:
                        print(f"  [probe failed: {str(e)[:60]}]"); _probed=True
                try:
                    f8=faithfulness_rankcorr(load_img(rp),grid=8)
                    if not np.isnan(f8): faiths.append(f8)
                    if len(faiths)>=30: break
                except Exception as e:
                    _fail+=1
                    if _fail<=2: print(f"  [faith fail: {str(e)[:60]}]")
            if CAM_OK and len(faiths)==0:
                print(f"  WARNING: faithfulness empty after {_fail} failures - check Grad-CAM on this arch")
            shutil.rmtree(fdir,ignore_errors=True)
    faith=float(np.mean(faiths)) if faiths else np.nan
    # calibrate
    use_calibration_env(); import metrics as metc; import calibration as cal
    p=sc.prob_fake.values.astype(float);y=sc.label.values.astype(int)
    g=sc.identity_id.values if 'identity_id' in sc.columns else None
    ci,ti,_=cal.leakage_safe_split(y,groups=g,calib_frac=0.5,seed=42)
    pcal,_=cal.fit_predict("hybrid",p[ci],y[ci],p[ti],switch_threshold_n=1000)
    auc=metc.roc_auc(p[ti],y[ti]);ece_cal=metc.ece(pcal,y[ti],15,'equal_mass')
    rf=reference_free_signals(p)
    row={'method':method,'AUC':round(auc,4),'ECE_cal':round(ece_cal,4),
         'faithfulness_rankcorr':round(faith,4) if not np.isnan(faith) else np.nan,
         **{k:round(v,4) for k,v in rf.items()},'n_faith':len(faiths)}
    pd.DataFrame([row]).to_csv(OUT,mode='a',header=not os.path.exists(OUT),index=False)
    print(f"  AUC={auc:.3f} ECE={ece_cal:.3f} faith={faith if np.isnan(faith) else round(faith,3)} ent={rf['entropy']:.3f} disp={rf['score_dispersion']:.3f}\n")

print("="*60)
print(pd.read_csv(OUT).to_string(index=False))

=== faceswap ===
  scored 25793
  AUC=0.993 ECE=0.010 faith=nan ent=0.088 disp=0.305

=== fomm ===
  scored 25898
  AUC=0.836 ECE=0.042 faith=nan ent=0.280 disp=0.360

=== fsgan ===
  scored 25221
  AUC=0.956 ECE=0.018 faith=nan ent=0.115 disp=0.329

=== wav2lip ===
  scored 26434
  AUC=0.641 ECE=0.099 faith=nan ent=0.370 disp=0.408

=== StyleGAN2 ===
  scored 33794
  AUC=0.821 ECE=0.023 faith=nan ent=0.226 disp=0.353

=== ddim ===
  scored 33794
  AUC=0.745 ECE=0.062 faith=nan ent=0.278 disp=0.384

=== simswap ===
  scored 25942
  AUC=0.960 ECE=0.019 faith=nan ent=0.119 disp=0.310

=== StyleGAN3 ===
  scored 33794
  AUC=0.774 ECE=0.029 faith=nan ent=0.260 disp=0.374

=== facevid2vid ===
  scored 25510
  AUC=0.838 ECE=0.044 faith=nan ent=0.259 disp=0.356

=== pirender ===
  scored 25805
  AUC=0.728 ECE=0.093 faith=nan ent=0.331 disp=0.394

=== DiT ===
  scored 33794
  AUC=0.561 ECE=0.143 faith=nan ent=0.360 disp=0.416

=== StyleGANXL ===
  scored 33794
  AUC=0.698 ECE=0.072 faith=nan e

## Cell 4 — Reference-free signals vs competence: do they generalize?

The key test: do the reference-free uncertainty signals track competence on THIS architecture
(and we'll compare to Xcep/EffNet)? If yes, they're a better, portable monitoring basis than KS.


In [ ]:
import pandas as pd, numpy as np
from scipy.stats import pearsonr
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"
A3=pd.read_csv(f"{REPO}/reports/calibration/unified_trust_signals_{ARCH_KEY}.csv").dropna(subset=['AUC','ECE_cal'])
RF=['entropy','score_dispersion','mean_max_conf','frac_uncertain']
print(f"=== {ARCH}: reference-free signal vs competence (n={len(A3)}) ===")
for s in RF:
    if s in A3 and A3[s].notna().all():
        r_auc,_=pearsonr(A3[s],A3['AUC']); r_ece,_=pearsonr(A3[s],A3['ECE_cal'])
        print(f"  {s:<18}: r(.,AUC)={r_auc:+.3f}  r(.,ECE)={r_ece:+.3f}")
print(f"\n  intrinsic: r(ECE,AUC)={pearsonr(A3['ECE_cal'],A3['AUC'])[0]:+.3f}", end="")
if A3['faithfulness_rankcorr'].notna().sum()>3:
    fa=A3.dropna(subset=['faithfulness_rankcorr'])
    print(f"  r(faith,AUC)={pearsonr(fa['faithfulness_rankcorr'],fa['AUC'])[0]:+.3f}")
else:
    print("  (faithfulness NaN - ViT, no conv map)")
print("\nCompare to Xception: entropy and dispersion SHOULD track competence if portable.")

## Cell 5 — Three-architecture intrinsic-signal single-factor + commit

In [ ]:
import pandas as pd, numpy as np, os
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
REPO = "/content/drive/MyDrive/CDTS_Research/deepfake-trust-research"

# load all three architectures
files={'Xception':'unified_trust_signals.csv','EfficientNet-B4':'unified_trust_signals_effnet.csv',
       ARCH:f'unified_trust_signals_{ARCH_KEY}.csv'}
def intrinsic_pc1(df):
    # intrinsic signals that generalized: ECE + faithfulness; add reference-free if present
    sigs=['ECE_cal','faithfulness_rankcorr']
    sigs=[s for s in sigs if s in df.columns and df[s].notna().all()]
    if len(sigs)<2: return None
    X=StandardScaler().fit_transform(df[sigs].values)
    p=PCA().fit(X);pc1=p.transform(X)[:,0];r,_=pearsonr(pc1,df['AUC'])
    if r<0: pc1=-pc1; r=-r
    return pc1, p.explained_variance_ratio_[0]*100, abs(r), sigs

results={}
for name,f in files.items():
    fp=f"{REPO}/reports/calibration/{f}"
    if not os.path.exists(fp): print(f"{name}: missing"); continue
    df=pd.read_csv(fp).dropna(subset=['AUC','ECE_cal'])
    res=intrinsic_pc1(df)
    if res: results[name]=(df,res)

n=len(results)
fig,axes=plt.subplots(1,n,figsize=(4.2*n,4.2))
if n==1: axes=[axes]
for ax,(name,(df,(pc1,var,r,sigs))) in zip(axes,results.items()):
    ax.scatter(df['AUC'],pc1,s=70,c='#34495E',edgecolor='white',zorder=3)
    b,a=np.polyfit(df['AUC'].values,pc1,1);xs=np.linspace(df['AUC'].min(),df['AUC'].max(),50)
    ax.plot(xs,a+b*xs,'k--',lw=1.5)
    ax.annotate(f"PC1={var:.0f}%  r={r:.2f}",xy=(.96,.06),xycoords='axes fraction',ha='right',fontsize=11,bbox=dict(boxstyle="round,pad=0.4",fc="white",ec="gray"))
    ax.set_xlabel("Competence (AUC)",fontsize=11);ax.set_ylabel("Intrinsic-signal PC1",fontsize=11)
    ax.set_title(f"{name}\n(intrinsic: {', '.join(sigs)})",fontsize=10);ax.grid(alpha=0.25)
plt.suptitle("Intrinsic-signal single factor (calibration + faithfulness) across architectures",fontsize=13,y=1.03)
plt.tight_layout()
out=f"{REPO}/figures/intrinsic_factor_three_arch.png"
plt.savefig(out,dpi=300,bbox_inches='tight');plt.show()
print("saved",out)
print("\n=== SUMMARY: intrinsic-signal single factor ===")
for name,(df,(pc1,var,r,sigs)) in results.items():
    print(f"  {name:<18}: PC1={var:.0f}%  r(PC1,AUC)={r:.2f}  (n={len(df)}, signals={sigs})")

import subprocess
os.chdir(REPO)
for f in [".gitconfig",".git-credentials"]:
    if os.path.exists(f"/content/drive/MyDrive/CDTS_Research/{f}"): subprocess.run(f'cp "/content/drive/MyDrive/CDTS_Research/{f}" /root/{f}', shell=True)
subprocess.run(f"git add reports/calibration/unified_trust_signals_{ARCH_KEY}.csv figures/intrinsic_factor_three_arch.png notebooks/NB14_third_arch_signals.ipynb", shell=True)
print("\n"+subprocess.run("git status --short",shell=True,capture_output=True,text=True).stdout)